In [7]:
import gurobipy as gp
from gurobipy import GRB

# ==============================================================================
# DEEP SKY ONE: TWO-TRANCHE DYNAMIC CAPITAL STRUCTURE OPTIMIZATION
# ==============================================================================
m = gp.Model("DeepSkyOne_Two_Tranche_Optimization")

# ------------------------------------------------------------------------------
# 1. Project & Fiscal Policy Inputs ($ Millions CAD)
# ------------------------------------------------------------------------------
phases = [1, 2, 3]
capex_gross = [231.609, 378.000, 1822.500]

# Clean Economy CCUS ITC: 60% rate on 90% eligible capture equipment
itc_effective_rate = 0.60 * 0.90  # 54.0% refundable rebate
net_capex = [c * (1.0 - itc_effective_rate) for c in capex_gross]

# SIF Net Zero Accelerator Non-dilutive Grants
sif_grants = [15.00, 10.00, 25.00]
net_capex_to_fund = [net_capex[p] - sif_grants[p] for p in range(3)]
T = 10

# ------------------------------------------------------------------------------
# 2. CFADS Decomposition: Contracted vs. Merchant VCM
# ------------------------------------------------------------------------------
# Contracted CFADS (Phases 1-2: 100% contracted; Phase 3: 195,375 t contracted)
cfads_c_p1 = (30375 * 450 - 30375 * 390) / 1e6  # $1.8225M
cfads_c_p2 = (72000 * 450 - 72000 * 315) / 1e6  # $9.7200M
cfads_c_p3 = (
    (135375 * 450 + 60000 * 500) - (195375 * 260)
) / 1e6  # $40.12125M/yr

cfads_c = [cfads_c_p1, cfads_c_p2] + [cfads_c_p3] * (T - 2)

# Merchant VCM CFADS (312,000 t residual volume in Phase 3)
cfads_m_base_p3 = (
    (312000 * 450) - (312000 * 260)
) / 1e6  # $59.2800M/yr @ $450/t
cfads_m_floor_p3 = (
    (312000 * 300) - (312000 * 260)
) / 1e6  # $12.4800M/yr @ $300/t floor

cfads_m_base = [0.0, 0.0] + [cfads_m_base_p3] * (T - 2)
cfads_m_floor = [0.0, 0.0] + [cfads_m_floor_p3] * (T - 2)
cfads_base_total = [cfads_c[t] + cfads_m_base[t] for t in range(T)]

# ------------------------------------------------------------------------------
# 3. Debt Terms & Discount Rates
# ------------------------------------------------------------------------------
gearing_cap_phase = [0.15, 0.25, 0.40]
min_sponsor_equity = 0.15
max_cgf_pref = 0.40

# Tranche A (Contracted): Lower spread due to contracted offtake certainty
r_dA = [0.0700, 0.0625] + [0.0500] * (T - 2)
dscr_cA = [1.40, 1.30] + [1.20] * (T - 2)

# Tranche B (Merchant): Higher spread for merchant volatility
r_dB = [0.0800, 0.0750] + [0.0650] * (T - 2)
dscr_m_base = [1.0, 1.0] + [1.75] * (T - 2)
dscr_m_floor = [1.0, 1.0] + [1.10] * (T - 2)

r_cgf_cash = 0.030  # 3.0% cash yield (Subordinated Convertible Pref)
r_e_foak = 0.180  # 18.0% FOAK Development Hurdle
r_e_infra = 0.105  # 10.5% Commercial Hurdle

df_equity_calls = [
    1.0,
    1.0 / (1.0 + r_e_foak),
    1.0 / ((1.0 + r_e_foak) * (1.0 + r_e_infra)),
]
df_dist = []
for t in range(1, T + 1):
  if t == 1:
    df_dist.append(1.0 / (1.0 + r_e_foak))
  elif t == 2:
    df_dist.append(1.0 / ((1.0 + r_e_foak) * (1.0 + r_e_infra)))
  else:
    df_dist.append(df_equity_calls[2] / ((1.0 + r_e_infra) ** (t - 2)))

# ------------------------------------------------------------------------------
# 4. Decision Variables
# ------------------------------------------------------------------------------
# Capital draws by phase (p = 0..2)
DA_draw = m.addVars(3, lb=0.0, name="TrancheA_Contracted_Draw")
DB_draw = m.addVars(3, lb=0.0, name="TrancheB_Merchant_Draw")
CGF_draw = m.addVars(3, lb=0.0, name="CGF_Pref_Draw")
E_draw = m.addVars(3, lb=0.0, name="Sponsor_Equity_Draw")

# Debt amortization & distribution schedules (t = 1..10)
DA = m.addVars(T + 1, lb=0.0, name="Balance_TrancheA")
PA = m.addVars(range(1, T + 1), lb=0.0, name="Principal_TrancheA")
IA = m.addVars(range(1, T + 1), lb=0.0, name="Interest_TrancheA")

DB = m.addVars(T + 1, lb=0.0, name="Balance_TrancheB")
PB = m.addVars(range(1, T + 1), lb=0.0, name="Principal_TrancheB")
IB = m.addVars(range(1, T + 1), lb=0.0, name="Interest_TrancheB")

Div = m.addVars(range(1, T + 1), lb=0.0, name="Sponsor_Distribution")

# ------------------------------------------------------------------------------
# 5. Constraints
# ------------------------------------------------------------------------------
for p in range(3):
  # Sources == Uses
  m.addConstr(
      DA_draw[p] + DB_draw[p] + CGF_draw[p] + E_draw[p] == net_capex_to_fund[p],
      name=f"Sources_Uses_Phase_{p+1}",
  )
  # Combined Gearing Cap
  m.addConstr(
      DA_draw[p] + DB_draw[p] <= gearing_cap_phase[p] * net_capex[p],
      name=f"Combined_Gearing_Cap_{p+1}",
  )
  # Min Sponsor Equity & Max CGF
  m.addConstr(
      E_draw[p] >= min_sponsor_equity * net_capex_to_fund[p],
      name=f"Min_Sponsor_Equity_{p+1}",
  )
  m.addConstr(
      CGF_draw[p] <= max_cgf_pref * net_capex_to_fund[p],
      name=f"CGF_Availability_{p+1}",
  )

# Merchant debt only available in Phase 3
m.addConstr(DB_draw[0] == 0.0, name="No_Merchant_Debt_P1")
m.addConstr(DB_draw[1] == 0.0, name="No_Merchant_Debt_P2")

# Boundary conditions
m.addConstr(DA[0] == 0.0, name="Initial_DebtA_Zero")
m.addConstr(DB[0] == 0.0, name="Initial_DebtB_Zero")
m.addConstr(DA[T] == 0.0, name="Terminal_DebtA_Maturity")
m.addConstr(DB[T] == 0.0, name="Terminal_DebtB_Maturity")

draw_map = {1: 0, 2: 1, 3: 2}

for t in range(1, T + 1):
  draw_idx = draw_map.get(t, None)
  new_draw_A = DA_draw[draw_idx] if draw_idx is not None else 0.0
  new_draw_B = DB_draw[draw_idx] if draw_idx is not None else 0.0

  # Tranche A Dynamics
  m.addConstr(IA[t] == r_dA[t - 1] * DA[t - 1], name=f"InterestA_Calc_{t}")
  m.addConstr(
      DA[t] == DA[t - 1] + new_draw_A - PA[t], name=f"BalanceA_Step_{t}"
  )
  m.addConstr(
      PA[t] + IA[t] <= cfads_c[t - 1] / dscr_cA[t - 1],
      name=f"TrancheA_DSCR_Limit_{t}",
  )

  # Tranche B Dynamics
  m.addConstr(IB[t] == r_dB[t - 1] * DB[t - 1], name=f"InterestB_Calc_{t}")
  m.addConstr(
      DB[t] == DB[t - 1] + new_draw_B - PB[t], name=f"BalanceB_Step_{t}"
  )
  if cfads_m_base[t - 1] > 0:
    m.addConstr(
        PB[t] + IB[t] <= cfads_m_base[t - 1] / dscr_m_base[t - 1],
        name=f"TrancheB_Base_DSCR_{t}",
    )
    m.addConstr(
        PB[t] + IB[t] <= cfads_m_floor[t - 1] / dscr_m_floor[t - 1],
        name=f"TrancheB_Floor_DSCR_{t}",
    )

  # Combined Debt Service Floor Protection
  m.addConstr(
      (PA[t] + IA[t]) + (PB[t] + IB[t])
      <= (cfads_c[t - 1] + cfads_m_floor[t - 1]) / 1.05,
      name=f"Combined_Floor_Coverage_{t}",
  )

  # Cash Flow Waterfall: Residual Cash to Equity
  cum_cgf = gp.quicksum(CGF_draw[p] for p in range(min(t, 3)))
  m.addConstr(
      Div[t]
      == cfads_base_total[t - 1]
      - (PA[t] + IA[t])
      - (PB[t] + IB[t])
      - (r_cgf_cash * cum_cgf),
      name=f"Waterfall_Distribution_{t}",
  )

# ------------------------------------------------------------------------------
# 6. Objective: Dynamic Coupling of CGF into Terminal Value
# ------------------------------------------------------------------------------
pv_equity_calls = gp.quicksum(E_draw[p] * df_equity_calls[p] for p in range(3))
pv_distributions = gp.quicksum(
    Div[t] * df_dist[t - 1] for t in range(1, T + 1)
)

remaining_years = 15
annuity_factor_terminal = (1.0 - (1.0 + r_e_infra) ** (-remaining_years)) / r_e_infra
total_cgf = gp.quicksum(CGF_draw[p] for p in range(3))

# Dynamic LinExpr: Unencumbered terminal equity accounts for actual CGF draws
pv_terminal_value = (
    (cfads_base_total[-1] * annuity_factor_terminal * df_dist[-1])
    - (r_cgf_cash * total_cgf * annuity_factor_terminal * df_dist[-1])
)

m.setObjective(
    pv_distributions + pv_terminal_value - pv_equity_calls, GRB.MAXIMIZE
)
m.optimize()

# ------------------------------------------------------------------------------
# 7. Reporting & Capital Stack Breakdown
# ------------------------------------------------------------------------------
if m.status == GRB.OPTIMAL:
  print("\n" + "=" * 80)
  print("DEEP SKY ONE: OPTIMAL TWO-TRANCHE CAPITAL STRUCTURE")
  print("=" * 80)
  for p in range(3):
    total_d = DA_draw[p].X + DB_draw[p].X
    print(f"\n--- Phase {p+1} Capital Stack ---")
    print(f"  Gross CapEx:                 ${capex_gross[p]:.2f}M")
    print(f"  CCUS ITC Bridge (54%):       ${capex_gross[p]*0.54:.2f}M")
    print(f"  SIF Grant:                   ${sif_grants[p]:.2f}M")
    print(f"  Net Capital to Fund:         ${net_capex_to_fund[p]:.2f}M")
    print(
        f"    • Tranche A (Contracted):  ${DA_draw[p].X:.2f}M"
        f" ({DA_draw[p].X/net_capex[p]*100:.1f}%)"
    )
    print(
        f"    • Tranche B (Merchant):    ${DB_draw[p].X:.2f}M"
        f" ({DB_draw[p].X/net_capex[p]*100:.1f}%)"
    )
    print(
        f"    • Total Senior/Mezz Debt:  ${total_d:.2f}M"
        f" ({total_d/net_capex[p]*100:.1f}% gearing)"
    )
    print(
        f"    • CGF Catalytic Pref:      ${CGF_draw[p].X:.2f}M"
        f" ({CGF_draw[p].X/net_capex_to_fund[p]*100:.1f}%)"
    )
    print(
        f"    • Sponsor Common Equity:   ${E_draw[p].X:.2f}M"
        f" ({E_draw[p].X/net_capex[p]*100:.1f}%)"
    )

  pv_out = sum(E_draw[p].X * df_equity_calls[p] for p in range(3))
  pv_in_ops = sum(Div[t].X * df_dist[t - 1] for t in range(1, T + 1))
  pv_tv = (
      cfads_base_total[-1] - 0.03 * sum(CGF_draw[p].X for p in range(3))
  ) * annuity_factor_terminal * df_dist[-1]

  print("\n" + "-" * 80)
  print("EQUITY VALUATION WATERFALL (CAD):")
  print(f"  • PV of Sponsor Equity Calls (t=0..2):         -${pv_out:.2f}M")
  print(f"  • PV of 8-Yr Operating Distributions (t=1..10): +${pv_in_ops:.2f}M")
  print(f"  • PV of Deleveraged Terminal Equity (t=11..25): +${pv_tv:.2f}M")
  print("-" * 80)
  print(f"Maximized Sponsor Equity NPV:                    +${m.ObjVal:.2f}M CAD")
  print("=" * 80)

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 11th Gen Intel(R) Core(TM) i3-1115G4 @ 3.00GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Optimize a model with 104 rows, 84 columns and 305 nonzeros (Max)
Model fingerprint: 0x73007dea
Model has 16 linear objective coefficients and an objective constant of -2.5358203328281070e+02
Coefficient statistics:
  Matrix range     [3e-02, 1e+00]
  Objective range  [8e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 8e+02]

Presolve removed 76 rows and 54 columns
Presolve time: 0.06s
Presolved: 28 rows, 30 columns, 148 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.8069065e+02   6.989620e+01   0.000000e+00      0s
      26    1.1683967e+02   0.000000e+00   0.000000e+00      0s

Solved in 26 iterations and 0.09 seconds (0.00 work units)
Optimal objective  1.16839674